In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [2]:
df = pd.read_csv("data/ames_housing.csv")
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.drop(index=[523, 1298])

X_train = train_df.drop(columns=['SalePrice', 'Id', 'GarageYrBlt', 'PoolArea'])
y_train = train_df['SalePrice']

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")

Shape of X_train: (1166, 77)
Shape of y_train: (1166,)


In [4]:
def make_preprocessor(X):
    numeric_cols = X.select_dtypes(include='number').columns.tolist()
    categorical_cols = X.select_dtypes(exclude='number').columns.tolist()
    numeric_cols.remove('MSSubClass')
    categorical_cols.append('MSSubClass')

    numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), 
                                    ('scaler', StandardScaler())])
    categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='None')),
                                     ('encoder', OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False))])
    preprocessor = ColumnTransformer([('num', numeric_pipeline, numeric_cols),
                                      ('cat', categorical_pipeline, categorical_cols)])
    return preprocessor


prep = make_preprocessor(X_train)
print(prep.fit_transform(X_train).shape)

(1166, 240)


In [7]:
def evaluate(model, X, y):
    pipe = Pipeline([('prep', make_preprocessor(X)), ('model', model)])

    mae_scores  = cross_val_score(pipe, X, y, cv=5, scoring='neg_mean_absolute_error')
    rmse_scores = cross_val_score(pipe, X, y, cv=5, scoring='neg_root_mean_squared_error')
    r2_scores   = cross_val_score(pipe, X, y, cv=5, scoring='r2')

    return -mae_scores.mean(), -rmse_scores.mean(), r2_scores.mean()


mae, rmse, r2 = evaluate(LinearRegression(), X_train, y_train)
print(mae, rmse, r2)

17207.463617697722 25395.88625441448 0.8896753334786069


In [12]:
models = {
    "Linear Regression": LinearRegression(),
    "KNN":               KNeighborsRegressor(),
    "Decision Tree":     DecisionTreeRegressor(random_state=42),
    "Random Forest":     RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

rows = []
for name, model in models.items():
    mae, rmse, r2 = evaluate(model, X_train, y_train)
    rows.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})

results = pd.DataFrame(rows).sort_values("MAE")
results.round(3)

,Model,MAE,RMSE,R2
4,Gradient Boosting,15862.926,24563.434,0.898
0,Linear Regression,17207.464,25395.886,0.890
3,Random Forest,17507.243,27526.435,0.872
1,KNN,21561.013,35084.766,0.793
2,Decision Tree,26596.503,40250.346,0.722


## Conclusions

All models use the same cleaned training data (1,166 houses), the same preprocessing and the same 5-fold split. None of them is tuned yet.

1. **Gradient Boosting is ahead (MAE $15,863, R² 0.898)**, but only 8% better than the Linear Regression baseline (MAE $17,207, R² 0.890). With single 5-fold averages, the gap between the top three models is small.
2. **Linear Regression is a strong baseline.** Prices in Ames depend mostly on size and quality in a fairly straight-line way, and the cleaning decisions (dropping `PoolArea` and two unfinished sales) removed the large errors that were hurting it.
3. **A single Decision Tree overfits** (R² 0.722). Averaging many trees in a Random Forest cuts the MAE from $26,597 to $17,507.
4. **KNN struggles** (R² 0.793): with 240 columns, mostly 0/1 flags, distances between houses carry little meaning.
5. **Next:** tune the best models and add XGBoost, LightGBM and CatBoost.